# NETRA: DistilBERT Tier-1 training on Kaggle

**Before you run (Kaggle settings, right sidebar):**
1. **Accelerator:** GPU (T4 x2 or P100).
2. **Internet:** On. The notebook downloads `distilbert-base-uncased` from Hugging Face.
3. **Add data:** two datasets.
   - `netra-code`: a Kaggle dataset containing the NETRA project folder (the one with `ml/`, `api/`, `tests/`).
   - `netra-raw-data`: a Kaggle dataset containing your raw corpora (the files that go in `data/raw/`).
4. Set `RAW_DATASET` in the setup cell to the path of your raw-data dataset.

Then **Save Version -> Save & Run All (Commit)**. It runs in the background. Outputs are in `/kaggle/working/outputs/`.


In [ ]:
import subprocess, sys, shutil, glob
from pathlib import Path

def run(cmd, cwd=None):
    """Stream a command's output. Returns the exit code instead of raising."""
    print("$", " ".join(str(c) for c in cmd))
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def must_run(cmd, cwd=None):
    rc = run(cmd, cwd)
    if rc != 0:
        raise RuntimeError(f"Command failed (exit {rc}). See output above.")

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Set Settings -> Accelerator -> GPU and run again.")
print("GPU:", torch.cuda.get_device_name(0))

must_run([sys.executable, "-m", "pip", "install", "-q", "tldextract==5.4.0"])


In [ ]:
# >>> Set this to your raw-data dataset folder (see /kaggle/input in the right sidebar) <<<
RAW_DATASET = "/kaggle/input/netra-raw-data"

def find_one(pattern):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"Nothing matches {pattern}. Is the dataset attached?")
    return Path(hits[0])

# Locate the code dataset by its data_pipeline.py, then copy it to a writable folder.
PROJECT_SRC = find_one("/kaggle/input/**/ml/data_pipeline.py").parents[1]
WORK = Path("/kaggle/working/NETRA")
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(PROJECT_SRC, WORK, ignore=shutil.ignore_patterns("__pycache__", "*.pt", "*.pkl"))
print("Project:", PROJECT_SRC, "->", WORK)

# Copy raw corpora into data/raw (the pipeline reads them from there).
RAW = WORK / "data" / "raw"
if not Path(RAW_DATASET).is_dir():
    raise FileNotFoundError(f"RAW_DATASET not found: {RAW_DATASET}")
shutil.copytree(RAW_DATASET, RAW, dirs_exist_ok=True)
print("Raw files:", sorted(p.name for p in RAW.iterdir())[:30])


## 1. Build unified.csv (data pipeline)

In [ ]:
must_run([sys.executable, WORK / "ml" / "data_pipeline.py"], cwd=WORK)


## 2. Check the dataset before spending GPU time

In [ ]:
import pandas as pd

UNIFIED = WORK / "data" / "processed" / "unified.csv"
df = pd.read_csv(UNIFIED, dtype=str, keep_default_na=False)
print("rows:", len(df))
print(df.groupby(["split", "label"]).size().to_string())
print("label_origin:", df["label_origin"].value_counts().to_dict())
print("url_only rows:", (df["url_only"] == "True").sum())

leak = df["body_text"].str.contains("Phishing URL").sum()
assert leak == 0, f"{leak} rows still contain the label-leaking prefix"
assert {"train", "val", "test_std"} <= set(df["split"]), "missing a required split"
print("Checks passed.")


## 3. Train DistilBERT

FP16 on the GPU. Checkpoint is chosen on validation F1 only. Test metrics are written to the config.

In [ ]:
must_run([
    sys.executable, WORK / "ml" / "train_distilbert.py",
    "--data", UNIFIED,
    "--epochs", "3",
    "--batch_size", "16",
    "--lr", "2e-5",
    "--max_length", "256",
    "--seed", "42",
    "--fp16",
], cwd=WORK)


## 4. Calibrate thresholds on validation probabilities

In [ ]:
must_run([sys.executable, WORK / "ml" / "calibrate_threshold.py", "--data", UNIFIED], cwd=WORK)


## 5. Results

In [ ]:
import json
cfg = json.loads((WORK / "ml/models/distilbert_config.json").read_text(encoding="utf-8"))
print("best epoch:", cfg["best_epoch"], "| val F1:", cfg["best_val_f1"])
for name, m in cfg.get("test_metrics", {}).items():
    print(f"{name}: n={m['n']} legit={m['n_legitimate']} malicious={m['n_malicious']} "
          f"P={m['precision']} R={m['recall']} F1={m['f1']} FPR={m['fpr']} PR-AUC={m['pr_auc']}")
print(json.dumps(json.loads((WORK / "ml/models/threshold_config.json").read_text()), indent=2))


## 6. End-to-end stress test

This is the acceptance gate. It must show `RESULT: PASSED`. Its exit code is printed but does not stop the notebook, so you can still save the model if it fails.

In [ ]:
rc = run([sys.executable, WORK / "tests" / "stress_test_tier1.py"], cwd=WORK)
print("stress test exit code:", rc, "(0 = passed)")


## 7. Save outputs

Everything in `/kaggle/working/outputs/` is kept with the notebook version.

In [ ]:
OUT = Path("/kaggle/working/outputs")
OUT.mkdir(parents=True, exist_ok=True)
MODELS = WORK / "ml" / "models"
shutil.copy2(MODELS / "distilbert_tier1.pt", OUT)
shutil.copy2(MODELS / "distilbert_config.json", OUT)
shutil.copy2(MODELS / "threshold_config.json", OUT)
shutil.copytree(MODELS / "distilbert_tokenizer", OUT / "distilbert_tokenizer", dirs_exist_ok=True)
shutil.copy2(UNIFIED, OUT)  # optional; remove if too large
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(OUT)}  {p.stat().st_size // 1024 // 1024} MB")
